# Iteration 2: a learned verifier for answer-or-skip

**Loss targeted:** the decision. Photos whose right place was found but which are skipped (too few inliers, no agreement), and the -20 risk.

**Hypothesis.** The raw inlier count does not transfer between data sets (Test_Iteration_3: 3-7 inliers are mostly right on official, mostly
wrong on NUDT). Properties of the fit itself should transfer better, because they describe *how* the matches agree, not how many there are: a right
match covers the photo (convex hull, spread), fits tightly (residual), uses most confident pairs (ratio), and needs no extra rotation or scale
(the query already applied the heading and scale guess, so a right fit has angle ~0 and scale ~1). Together with agreement and DINO z they may
separate right from wrong at low inlier counts on both data sets.

**Input:** one row per photo and decision step (the choice over the 6 standard queries, and over all 27), features from the stored LoFTR matches
(`train/checks/verifier_features.py`). **Output:** P(wrong = error > 8 sigma). **Replaces:** the bar 8 / agreement / bar 11 rule (or is added to
it). **Training data:** official + NUDT 101 (out-of-fold, folds grouped by photo). **Test:** NUDT 202 and the fresh NUDT 404, never used to choose.
**Leakage risk:** the label uses the truth only; flight height and map level are left out of the features, because the evaluation heights differ.
**Cheapest test:** logistic regression and a shallow gradient-boosting model, no network.

In [1]:
import os, sys, time
from pathlib import Path
import numpy as np, pandas as pd
ROOT = Path.cwd()
while not (ROOT / "single_photo_pipeline.py").exists():
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT)); sys.path.insert(0, str(ROOT / "src")); os.chdir(ROOT)
pd.set_option("display.width", 220); pd.set_option("display.max_columns", 30)
from train.checks.rich_dump import load_rich
from train.checks.verifier_features import decision_rows
from train.checks.verifier import FEATURES, feature_matrix, oof_probability, fit_predict, pick_threshold, apply_rule, labels
from train.checks.replay import score, paired_bootstrap
DUMPS = ROOT / "tests/Test_Iteration_6/results/rich_dump"
OUT = ROOT / "tests/Test_Iteration_6/results"
SET_NAMES = {"official": "official", "nudt101": "nudt_seed101", "nudt202": "nudt_seed202", "nudt404": "nudt_fresh404"}
CHOOSE_ON, REPORT_ON = ["official", "nudt101"], ["nudt202", "nudt404"]

## 1. Decision rows (cached per set)

In [2]:
tables = {}
for name, folder in SET_NAMES.items():
    cache = OUT / ("it2_rows_%s.csv" % name)
    files = sorted((DUMPS / folder).glob("photo_*.npz")) if (DUMPS / folder).exists() else []
    if cache.exists() and len(pd.read_csv(cache)) == 2 * len(files):
        tables[name] = pd.read_csv(cache); continue
    if not files:
        continue
    start = time.time(); rows = []
    for p in files:
        rows += decision_rows(load_rich(p), int(p.stem.split("_")[1]), name)
    tables[name] = pd.DataFrame(rows); tables[name].to_csv(cache, index=False)
    print(name, len(files), "photos, %.0f s" % (time.time() - start))
rows_all = pd.concat(tables.values(), ignore_index=True)
print({k: len(v) // 2 for k, v in tables.items()})

nudt404 200 photos, 9 s
{'official': 379, 'nudt101': 300, 'nudt202': 200, 'nudt404': 200}


## 2. Does each feature separate right from wrong the same way on every set?

AUC of each feature for "wrong" among the hard rows only: choices with 3 to 10 inliers (where the bar decides). 0.5 = no information; a feature
that is > 0.5 on one set and < 0.5 on another cannot transfer.

In [3]:
from sklearn.metrics import roc_auc_score
X_all = pd.DataFrame(feature_matrix(rows_all), columns=FEATURES, index=rows_all.index)
hard = rows_all.has_choice & rows_all.inliers.between(3, 10)
res = {}
for name in tables:
    sel = hard & (rows_all.data == name)
    y = labels(rows_all[sel])
    res[name] = {f: (roc_auc_score(y, X_all.loc[sel, f]) if 0 < y.sum() < len(y) else np.nan) for f in FEATURES}
    res[name]["rows (wrong)"] = "%d (%d)" % (sel.sum(), y.sum())
display(pd.DataFrame(res).round(2))

,official,nudt101,nudt202,nudt404
log_inliers,0.335659,0.154839,0.212121,0.231092
z,0.015504,0.095161,0.272727,0.042017
rank,0.888372,0.814516,0.570707,0.920168
agree,0.143411,0.16129,0.388889,0.246499
agree_any,0.143411,0.170968,0.383838,0.266106
log_runner_far,0.687597,0.53629,0.545455,0.471989
ratio,0.368992,0.445161,0.469697,0.331933
fit_scale_dev,0.609302,0.703226,0.616162,0.481793
fit_angle,0.823256,0.783871,0.555556,0.436975
resid,0.596899,0.504839,0.40404,0.434174


## 3. Train on official + NUDT 101, choose the cut there, report on NUDT 202 / 303

* **today**: bar 8 or agreement on 6 queries, else bar 11 on 27 (the current best rule).
* **verifier (replace)**: answer the 6-query choice if P(wrong) < t_std, else the 27-query choice if P(wrong) < t_wide.
* **today + verifier (add)**: today's rule, plus any extra answer the verifier allows, at either step.
* **today + verifier (late)**: today's full rule first (6 queries, then the 27-query escalation); the verifier only answers photos today's rule
  still skips. Added after a first look at NUDT 202 (photo 377: the add mode answered a weak 6-query choice wrongly and so lost the escalation,
  which would have been right). **NUDT 202 is therefore not unseen for this variant; the fresh NUDT 404 is its clean test.**
* Cuts: the largest with 0 out-of-fold wrong answers on the choosing sets (`x1.0`), and half of it (`x0.5`) as a safety margin.

In [4]:
choose_rows = rows_all[rows_all.data.isin(CHOOSE_ON)].reset_index(drop=True)
report_rows = rows_all[rows_all.data.isin(REPORT_ON)].reset_index(drop=True)
summary = []; per_photo = {}
for kind in ("logistic", "gbm"):
    p_oof = oof_probability(choose_rows, kind)
    t_std, t_wide = pick_threshold(choose_rows, p_oof, "std6"), pick_threshold(choose_rows, p_oof, "wide27")
    p_rep, model = fit_predict(choose_rows, report_rows, kind)
    print(kind, "cuts: std6 %.3f  wide27 %.3f" % (t_std, t_wide))
    for factor in (1.0, 0.5):
        for mode, keep, late in (("replace", False, False), ("add", True, False), ("late", True, True)):
            for part, rows_, p_ in (("choose (OOF)", choose_rows, p_oof), ("report", report_rows, p_rep)):
                today = apply_rule(rows_, p_, 0.0, 0.0, keep_today=True)
                new = apply_rule(rows_, p_, factor * t_std, factor * t_wide, keep_today=keep, late=late)
                for data in new.data.unique():
                    a, b = today[today.data == data], new[new.data == data]
                    g = paired_bootstrap(a.points.to_numpy(), b.points.to_numpy())
                    summary.append(dict(model=kind, cut=factor, mode=mode, set=data, part=part, today=round(score(a.points), 2),
                                        new=round(score(b.points), 2), gain="%+.2f [%+.2f, %+.2f]" % g, wrong_today=int((a.points < 0).sum()),
                                        wrong_new=int((b.points < 0).sum()), answered_new=round((b.stage != "skip").mean(), 3)))
                    per_photo[(kind, factor, mode, data)] = b
summary = pd.DataFrame(summary)
display(summary)
summary.to_csv(OUT / "it2_verifier_summary.csv", index=False)

logistic cuts: std6 0.256  wide27 0.350


gbm cuts: std6 0.283  wide27 0.029


,model,cut,mode,set,part,today,new,gain,wrong_today,wrong_new,answered_new
0,logistic,1.0,replace,official,choose (OOF),53.54,56.61,"+3.07 [+1.61, +4.62]",0,0,0.966
1,logistic,1.0,replace,nudt101,choose (OOF),55.56,55.04,"-0.52 [-1.40, +0.20]",0,0,0.920
2,logistic,1.0,replace,nudt202,report,57.36,54.66,"-2.70 [-5.10, -0.90]",0,1,0.930
3,logistic,1.0,replace,nudt404,report,53.43,56.10,"+2.67 [+0.18, +5.67]",4,0,0.945
4,logistic,1.0,add,official,choose (OOF),53.54,57.02,"+3.48 [+2.18, +4.91]",0,0,0.974
5,logistic,1.0,add,nudt101,choose (OOF),55.56,55.76,"+0.20 [+0.00, +0.60]",0,0,0.933
6,logistic,1.0,add,nudt202,report,57.36,56.46,"-0.90 [-2.70, +0.00]",0,1,0.965
7,logistic,1.0,add,nudt404,report,53.43,53.79,"+0.36 [+0.00, +0.90]",4,4,0.970
8,logistic,1.0,late,official,choose (OOF),53.54,57.15,"+3.61 [+2.28, +5.07]",0,0,0.974
9,logistic,1.0,late,nudt101,choose (OOF),55.56,55.76,"+0.20 [+0.00, +0.60]",0,0,0.933


## 3b. Without count features

The same as section 3 (logistic, cut x1.0, late mode), but the model never sees a count (raw inliers, pairs, far-away runner-up inliers): only
DINO z, coarse rank, agreement and the shape of the fit. If the gain holds, it does not rest on the inlier scale that fails to transfer between
data sets.

In [5]:
COUNT_FREE = [f for f in FEATURES if f not in ("log_inliers", "log_pairs", "log_runner_far")]
rows = []
for feats, label in ((FEATURES, "all features"), (COUNT_FREE, "count-free")):
    p_oof = oof_probability(choose_rows, "logistic", features=feats)
    t_std, t_wide = pick_threshold(choose_rows, p_oof, "std6"), pick_threshold(choose_rows, p_oof, "wide27")
    p_rep, _ = fit_predict(choose_rows, report_rows, "logistic", features=feats)
    for part, rows_, p_ in (("choose (OOF)", choose_rows, p_oof), ("report", report_rows, p_rep)):
        today = apply_rule(rows_, p_, 0.0, 0.0, keep_today=True); new = apply_rule(rows_, p_, t_std, t_wide, keep_today=True, late=True)
        for data in new.data.unique():
            a, b = today[today.data == data], new[new.data == data]
            g = paired_bootstrap(a.points.to_numpy(), b.points.to_numpy())
            rows.append(dict(features=label, set=data, part=part, today=round(score(a.points), 2), new=round(score(b.points), 2),
                             gain="%+.2f [%+.2f, %+.2f]" % g, wrong_new=int((b.points < 0).sum())))
display(pd.DataFrame(rows))

,features,set,part,today,new,gain,wrong_new
0,all features,official,choose (OOF),53.54,57.15,"+3.61 [+2.28, +5.07]",0
1,all features,nudt101,choose (OOF),55.56,55.76,"+0.20 [+0.00, +0.60]",0
2,all features,nudt202,report,57.36,57.36,"+0.00 [+0.00, +0.00]",0
3,all features,nudt404,report,53.43,53.79,"+0.36 [+0.00, +0.90]",4
4,count-free,official,choose (OOF),53.54,56.90,"+3.36 [+2.06, +4.78]",0
5,count-free,nudt101,choose (OOF),55.56,55.76,"+0.20 [+0.00, +0.60]",0
6,count-free,nudt202,report,57.36,57.36,"+0.00 [+0.00, +0.00]",0
7,count-free,nudt404,report,53.43,53.61,"+0.18 [+0.00, +0.54]",4


## 4. Transfer in both directions (the hard test)

Train on NUDT only (101) and score official; train on official only and score NUDT 101 / 202 / 303. Cut chosen out-of-fold on the training set.

In [6]:
rows = []
for train_sets, test_sets in ((["nudt101"], ["official"]), (["nudt101", "nudt202"], ["official"]), (["official"], ["nudt101", "nudt202", "nudt404"])):
    tr = rows_all[rows_all.data.isin(train_sets)].reset_index(drop=True)
    te = rows_all[rows_all.data.isin(test_sets)].reset_index(drop=True)
    if len(tr) == 0 or len(te) == 0:
        continue
    for kind in ("logistic", "gbm"):
        p_oof = oof_probability(tr, kind)
        t_std, t_wide = pick_threshold(tr, p_oof, "std6"), pick_threshold(tr, p_oof, "wide27")
        p_te, _ = fit_predict(tr, te, kind)
        for mode, keep, late in (("replace", False, False), ("add", True, False), ("late", True, True)):
            today = apply_rule(te, p_te, 0.0, 0.0, keep_today=True); new = apply_rule(te, p_te, t_std, t_wide, keep_today=keep, late=late)
            for data in new.data.unique():
                a, b = today[today.data == data], new[new.data == data]
                g = paired_bootstrap(a.points.to_numpy(), b.points.to_numpy())
                rows.append(dict(train="+".join(train_sets), test=data, model=kind, mode=mode, today=round(score(a.points), 2), new=round(score(b.points), 2),
                                 gain="%+.2f [%+.2f, %+.2f]" % g, wrong_new=int((b.points < 0).sum())))
transfer = pd.DataFrame(rows); display(transfer); transfer.to_csv(OUT / "it2_verifier_transfer.csv", index=False)

,train,test,model,mode,today,new,gain,wrong_new
0,nudt101,official,logistic,replace,53.54,55.12,"+1.58 [+0.13, +3.04]",1
1,nudt101,official,logistic,add,53.54,55.54,"+1.99 [+0.66, +3.36]",1
2,nudt101,official,logistic,late,53.54,56.07,"+2.53 [+1.42, +3.80]",0
3,nudt101,official,gbm,replace,53.54,55.06,"+1.52 [+0.16, +2.85]",0
4,nudt101,official,gbm,add,53.54,55.85,"+2.31 [+1.20, +3.48]",0
5,nudt101,official,gbm,late,53.54,55.85,"+2.31 [+1.20, +3.48]",0
6,nudt101+nudt202,official,logistic,replace,53.54,55.95,"+2.41 [+0.98, +3.86]",1
7,nudt101+nudt202,official,logistic,add,53.54,56.04,"+2.50 [+1.11, +3.96]",1
8,nudt101+nudt202,official,logistic,late,53.54,56.64,"+3.10 [+1.84, +4.46]",0
9,nudt101+nudt202,official,gbm,replace,53.54,52.08,"-1.46 [-2.66, -0.32]",0


## 5. Where the remaining marks are (today's rule), by height band and stage

The ceiling a decision could reach: answer exactly the photos whose chosen place is within 4 sigma (6-query choice, or 27-query choice when the
6-query one is not).

In [7]:
today_all = apply_rule(rows_all, np.full(len(rows_all), np.nan), 0.0, 0.0, keep_today=True)
s6 = rows_all[rows_all.step == "std6"].set_index(["data", "photo"]); w27 = rows_all[rows_all.step == "wide27"].set_index(["data", "photo"])
from train.checks.replay import points as pts
best = [max(pts(s6.loc[k].err_sigma) if s6.loc[k].has_choice else 0, pts(w27.loc[k].err_sigma) if w27.loc[k].has_choice else 0, 0) for k in s6.index]
today_all["ceiling"] = best
today_all["band"] = pd.cut(today_all.height, [0, 150, 250, 400, 5000], labels=["<=150", "150-250", "250-400", ">400"])
tab = today_all.groupby(["data", "band"], observed=True).agg(photos=("points", "size"), score=("points", lambda s: round(score(s), 2)),
                                                              ceiling=("ceiling", lambda s: round(score(s), 2)))
display(tab)

photos  score  ceiling
data     band                           
nudt101  150-250       4   0.00     4.50
         250-400     296  56.31    56.92
nudt202  150-250       1   0.00     0.00
         250-400     199  57.65    58.43
nudt404  150-250       4   0.00     0.00
         250-400     196  55.74    58.07
official <=150       167  50.59    57.84
         150-250      59  57.76    58.78
         250-400     153  55.14    57.96